# ETL pipeline assignment

### Bronze Layer

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ASSIGNMENT;
USE CATALOG ASSIGNMENT;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS BRONZE;

In [0]:
%sql
USE CATALOG ASSIGNMENT;
USE SCHEMA BRONZE;

### table1: media_customer_reviews

In [0]:
%sql
CREATE OR REPLACE TABLE BRONZE.media_customer_reviews
USING DELTA AS
select * from parquet.`/Volumes/exercise/exercise/exercise/Bakehouse_Dataset/media_customer_reviews.parquet`;

In [0]:
%sql
use catalog `assignment`; 
select * from `bronze`.`media_customer_reviews` limit 5;

In [0]:
%sql
use catalog `assignment`; 
select new_id, count(*) as review_count from `bronze`.`media_customer_reviews` 
group by new_id
having review_count > 1
order by review_count desc;

### table2: media_gold_reviews_chunked

In [0]:
%sql
CREATE OR REPLACE TABLE BRONZE.media_gold_reviews_chunked
USING DELTA AS
select * from parquet.`/Volumes/exercise/exercise/exercise/Bakehouse_Dataset/media_gold_reviews_chunked.parquet`;

In [0]:
%sql
select * from bronze.media_gold_reviews_chunked limit 5;

In [0]:
%sql
use catalog `assignment`; 
select franchiseID, chunk_id, review_date, count(*) as review_count from `bronze`.`media_gold_reviews_chunked` 
group by franchiseID, chunk_id, review_date, review_uri
having review_count > 1
order by review_count desc;

### table3: sales_customers

In [0]:
%sql
CREATE OR REPLACE TABLE BRONZE.sales_customers
USING DELTA AS
select * from parquet.`/Volumes/exercise/exercise/exercise/Bakehouse_Dataset/sales_customers.parquet`;

In [0]:
%sql
use catalog `assignment`; 
select * from `bronze`.`sales_customers` limit 5;

In [0]:
%sql
use catalog `assignment`; 
select customerID, count(*) as row_count from `bronze`.`sales_customers`
group by customerID
having row_count > 1
order by row_count desc;

### table4: sales_franchises

In [0]:
%sql
CREATE OR REPLACE TABLE BRONZE.sales_franchises
USING DELTA AS
select * from parquet.`/Volumes/exercise/exercise/exercise/Bakehouse_Dataset/sales_franchises.parquet`;

In [0]:
%sql
use catalog `assignment`; 
select * from `bronze`.`sales_franchises` limit 5;

In [0]:
%sql
use catalog `assignment`; 
select franchiseID, count(*) as row_count from `bronze`.`sales_franchises` 
group by franchiseID
having row_count > 1
order by row_count desc;

### table5: sales_suppliers

In [0]:
%sql
CREATE OR REPLACE TABLE BRONZE.sales_suppliers
USING DELTA AS
select * from parquet.`/Volumes/exercise/exercise/exercise/Bakehouse_Dataset/sales_suppliers.parquet`;

In [0]:
%sql
use catalog `assignment`; 
select * from `bronze`.`sales_suppliers` limit 5;

In [0]:
%sql
use catalog `assignment`; 
select supplierID, count(*) as row_count from `bronze`.`sales_suppliers`
group by supplierID
having row_count > 1
order by row_count desc;

### table6: sales_transactions

In [0]:
%sql
CREATE OR REPLACE TABLE BRONZE.sales_transactions
USING DELTA AS
select * from parquet.`/Volumes/exercise/exercise/exercise/Bakehouse_Dataset/sales_transactions.parquet`;

In [0]:
%sql
use catalog `assignment`; 
select * from `bronze`.`sales_transactions` limit 5;

In [0]:
%sql
use catalog `assignment`; 
select transactionID, count(*) as row_count from `bronze`.`sales_transactions` 
group by transactionID
having row_count > 1
order by row_count desc;

## Silver Layer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS SILVER;
USE SCHEMA SILVER;

In [0]:
%sql
USE CATALOG ASSIGNMENT;
USE SCHEMA SILVER;

### table1: media_customer_reviews

In [0]:
%sql
-- 3000037	
update bronze.media_customer_reviews 
set franchiseID = 3000037 
where new_id = 1;

In [0]:
%sql
-- drop table silver.media_customer_reviews;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Step 1: Read the Bronze table
bronze_df = spark.read.table("bronze.media_customer_reviews") 

# Step 2: Check if the Silver table exists
silver_table_name = "silver.media_customer_reviews"  
is_table_exists = spark.catalog.tableExists(silver_table_name)

# If the Silver table doesn't exist, create it using the Bronze table data
if not is_table_exists:
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    silver_table = DeltaTable.forName(spark, silver_table_name)

# Step 3: Perform the SCD1 Merge operation (update or insert records)
merge_result = silver_table.alias("silver").merge(
    bronze_df.alias("bronze"),
    "silver.new_id = bronze.new_id"
).whenMatchedUpdateAll(condition="""
        silver.review != bronze.review OR 
        silver.franchiseID != bronze.franchiseID OR 
        silver.review_date != bronze.review_date
    """) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%sql
use catalog `assignment`; 
select * from `silver`.`media_customer_reviews` where new_id=1;

### table2: media_gold_reviews_chunked

In [0]:
%sql
-- drop table assignment.silver.media_gold_reviews_chunked;

In [0]:
%sql
-- drop table assignment.silver.media_gold_reviews_chunked;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Step 1: Read the Bronze table
bronze_df = spark.read.table("bronze.media_gold_reviews_chunked") 

# Step 2: Check if the Silver table exists
silver_table_name = "silver.media_gold_reviews_chunked" 
is_table_exists = spark.catalog.tableExists(silver_table_name)

# If the Silver table doesn't exist, create it using the Bronze table data
if not is_table_exists: 
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    silver_table = DeltaTable.forName(spark, silver_table_name)

# Step 3: Perform the SCD1 Merge operation (update or insert records)
merge_result = silver_table.alias("silver").merge(
    bronze_df.alias("bronze"),
    "silver.chunk_id = bronze.chunk_id AND silver.franchiseID = bronze.franchiseID AND silver.review_date = bronze.review_date"
).whenMatchedUpdateAll(condition="""
        silver.chunked_text != bronze.chunked_text OR 
        silver.review_uri != bronze.review_uri
    """) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%sql
use catalog `assignment`; 
select * from `silver`.`media_gold_reviews_chunked` limit 5;

### table3: sales_customers

In [0]:
%sql
-- drop table silver.sales_customers;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Step 1: Read the Bronze table
bronze_df = spark.read.table("bronze.sales_customers") 

# Step 2: Check if the Silver table exists
silver_table_name = "silver.sales_customers" 
is_table_exists = spark.catalog.tableExists(silver_table_name)

# If the Silver table doesn't exist, create it using the Bronze table data
if not is_table_exists: 
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    silver_table = DeltaTable.forName(spark, silver_table_name)

# Step 3: Perform the SCD1 Merge operation (update or insert records)
merge_result = silver_table.alias("silver").merge(
    bronze_df.alias("bronze"),
    "silver.customerID = bronze.customerID"
).whenMatchedUpdateAll(condition="""
        silver.first_name != bronze.first_name OR
        silver.last_name != bronze.last_name OR
        silver.email_address != bronze.email_address OR
        silver.phone_number != bronze.phone_number OR
        silver.address != bronze.address OR
        silver.city != bronze.city OR
        silver.state != bronze.state OR
        silver.country != bronze.country OR
        silver.continent != bronze.continent OR
        silver.postal_zip_code != bronze.postal_zip_code OR
        silver.gender != bronze.gender
    """) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%sql
use catalog `assignment`; 
select * from `silver`.`sales_customers` limit 5;

### table4: sales_franchises

In [0]:
%sql
-- drop table silver.sales_franchises;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Step 1: Read the Bronze table
bronze_df = spark.read.table("bronze.sales_franchises") 

# Step 2: Check if the Silver table exists
silver_table_name = "silver.sales_franchises" 
is_table_exists = spark.catalog.tableExists(silver_table_name)

# If the Silver table doesn't exist, create it using the Bronze table data
if not is_table_exists: 
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    silver_table = DeltaTable.forName(spark, silver_table_name)

# Step 3: Perform the SCD1 Merge operation (update or insert records)
merge_result = silver_table.alias("silver").merge(
    bronze_df.alias("bronze"),
    "silver.franchiseID = bronze.franchiseID"
).whenMatchedUpdateAll(condition="""
        silver.name != bronze.name OR
        silver.city != bronze.city OR
        silver.district != bronze.district OR
        silver.zipcode != bronze.zipcode OR 
        silver.country != bronze.country OR
        silver.size != bronze.size OR
        silver.longitude != bronze.longitude OR
        silver.latitude != bronze.latitude OR 
        silver.supplierID != bronze.supplierID
    """) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%sql
use catalog `assignment`; 
select * from `silver`.`sales_franchises` limit 5;

### table5: sales_suppliers

In [0]:
%sql
-- drop table silver.sales_suppliers;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Step 1: Read the Bronze table
bronze_df = spark.read.table("bronze.sales_suppliers") 

# Step 2: Check if the Silver table exists
silver_table_name = "silver.sales_suppliers" 
is_table_exists = spark.catalog.tableExists(silver_table_name)

# If the Silver table doesn't exist, create it using the Bronze table data
if not is_table_exists: 
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    silver_table = DeltaTable.forName(spark, silver_table_name)

# Step 3: Perform the SCD1 Merge operation (update or insert records)
merge_result = silver_table.alias("silver").merge(
    bronze_df.alias("bronze"),
    "silver.supplierID = bronze.supplierID"
).whenMatchedUpdateAll(condition="""
        silver.name != bronze.name OR
        silver.ingredient != bronze.ingredient OR
        silver.continent != bronze.continent OR
        silver.city != bronze.city OR 
        silver.district != bronze.district OR
        silver.size != bronze.size OR
        silver.longitude != bronze.longitude OR
        silver.latitude != bronze.latitude OR 
        silver.approved != bronze.approved
    """) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%sql
use catalog `assignment`; 
select * from `silver`.`sales_suppliers` limit 5;

### table6: sales_transactions

In [0]:
%sql
-- drop table silver.sales_transactions;

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

# Initialize Spark session
spark = SparkSession.builder.getOrCreate()

# Step 1: Read the Bronze table
bronze_df = spark.read.table("bronze.sales_transactions") 

# Step 2: Check if the Silver table exists
silver_table_name = "silver.sales_transactions" 
is_table_exists = spark.catalog.tableExists(silver_table_name)

# If the Silver table doesn't exist, create it using the Bronze table data
if not is_table_exists: 
    bronze_df.write.format("delta").mode("overwrite").saveAsTable(silver_table_name)
    silver_table = DeltaTable.forName(spark, silver_table_name)

# Step 3: Perform the SCD1 Merge operation (update or insert records)
merge_result = silver_table.alias("silver").merge(
    bronze_df.alias("bronze"),
    "silver.transactionID = bronze.transactionID"
).whenMatchedUpdateAll(condition="""
        silver.customerID != bronze.customerID OR
        silver.franchiseID != bronze.franchiseID OR
        silver.dateTime != bronze.dateTime OR
        silver.product != bronze.product OR 
        silver.quantity != bronze.quantity OR
        silver.unitPrice != bronze.unitPrice OR
        silver.totalPrice != bronze.totalPrice OR
        silver.paymentMethod != bronze.paymentMethod OR 
        silver.cardNumber != bronze.cardNumber
    """) \
.whenNotMatchedInsertAll() \
.execute()

In [0]:
%sql
use catalog `assignment`; 
select * from `silver`.`sales_transactions` limit 5;

## Gold Layer

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS GOLD;

In [0]:
%sql
USE CATALOG ASSIGNMENT;
USE SCHEMA GOLD;

### 1.	Get the most sold products to identify the top-selling items.

In [0]:
from pyspark.sql.functions import sum, col, current_date, date_sub, count

# Load sales data (replace 'sales_data' with your actual table name)
df = spark.read.table("silver.sales_transactions")

# Optional: Filter the data
df_filtered = df.filter(col("product").isNotNull())

# Aggregate total quantity sold per product
top_selling_items = df_filtered.groupBy("product") \
    .agg(count("*").alias("total_product_sold")) \
    .orderBy(col("total_product_sold").desc())

# Show top 10 best-selling products
top_selling_items.show(10)

# Save result as a table 
top_selling_items.write.format("delta").mode("overwrite").saveAsTable("gold.top_selling_products")


In [0]:
%sql
select * from gold.top_selling_products;

### 2.	Find which suppliers provide ingredients to the most franchises.

In [0]:
# table: sales_franchises
from pyspark.sql.functions import sum, col, current_date, date_sub, count, collect_set

# Load sales data (replace 'sales_data' with your actual table name)
df = spark.read.table("silver.sales_franchises")

# Optional: Filter the data
df_filtered = df.filter(col("supplierID").isNotNull())

# Aggregate total quantity sold per product
top_selling_items = df_filtered.groupBy("franchiseID" ,"supplierID") \
    .agg(count("*").alias("supplier_franchise_count")) \
    .orderBy(col("supplier_franchise_count").desc())

# Show top 10 best-selling products
top_selling_items.show(10)

# Save result as a table 
top_selling_items.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("gold.supplier_franchise_details")


In [0]:
%sql
select * from gold.supplier_franchise_details limit 5;

### 3.	Get total sales per month.

In [0]:
---------------------------